# IIC-3670 NLP UC

## Actividad en clase

Vamos a crear un agente experto en un tema, haciendo RAG, de manera que el LLM entregue respuestas sensitivas al contexto.

- Instale **faiss**.
- Revise la guía de anotación disponible en: https://knowledge4policy.ec.europa.eu/text-mining/news-categorization-framing-persuasion-techniques-annotation-guidelines_en
- Escoja un problema (yo mostré oversimplification en persuasion techniques). 
- Usando la guía de anotación, define un example bank con al menos 5 ejemplos. 
- Implemente el agente experto en su problema, use faiss para que haga few shot con k=3.
- Corra un ejemplo de la guía de anotación y muestre el razonamiento y resultado del agente. Comente si coincide con el resultado de la guía.
- Cuando termine, me avisa para entregarle una **L (logrado)**.
- Recuerde que las L otorgan un bono en la nota final de la asignatura.


***Tiene hasta el final de la clase.***

In [1]:
import sys, platform, subprocess
print("Python:", platform.python_version())      # p.ej., 3.11.9
print("Version info:", sys.version_info)         # tuple detallada
print("Executable:", sys.executable)             # ruta del intérprete
subprocess.run([sys.executable, "-m", "pip", "--version"])  # pip correspondiente

Python: 3.10.12
Version info: sys.version_info(major=3, minor=10, micro=12, releaselevel='final', serial=0)
Executable: /usr/bin/python3
pip 25.2 from /home/marcelo/.local/lib/python3.10/site-packages/pip (python 3.10)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', '--version'], returncode=0)

In [2]:
import dspy
import importlib.metadata

info = importlib.metadata.metadata("dspy")

print("Nombre:", info["Name"])
print("Versión:", info["Version"])
print("Email autor:", info["Author-email"])
print("Licencia:", info["License"])
print("Resumen:", info["Summary"])

Nombre: dspy
Versión: 2.6.27
Email autor: Omar Khattab <okhattab@stanford.edu>
Licencia: MIT License
Resumen: DSPy


In [3]:
info = importlib.metadata.metadata("faiss-cpu")

print("Nombre:", info["Name"])
print("Versión:", info["Version"])
print("Email autor:", info["Author-email"])
print("Licencia:", info["License"])
print("Resumen:", info["Summary"])

Nombre: faiss-cpu
Versión: 1.8.0
Email autor: Kota Yamaguchi <yamaguchi_kota@cyberagent.co.jp>
Licencia: MIT License
Resumen: A library for efficient similarity search and clustering of dense vectors.


In [4]:
info = importlib.metadata.metadata("sentence-transformers")

print("Nombre:", info["Name"])
print("Versión:", info["Version"])
print("Email autor:", info["Author-email"])
print("Licencia:", info["License"])
print("Resumen:", info["Summary"])

Nombre: sentence-transformers
Versión: 2.2.2
Email autor: info@nils-reimers.de
Licencia: Apache License 2.0
Resumen: Multilingual text embeddings


In [5]:
info = importlib.metadata.metadata("huggingface_hub")

print("Nombre:", info["Name"])
print("Versión:", info["Version"])
print("Email autor:", info["Author-email"])
print("Licencia:", info["License"])
print("Resumen:", info["Summary"])

Nombre: huggingface-hub
Versión: 0.16.4
Email autor: julien@huggingface.co
Licencia: Apache
Resumen: Client library to download and publish models, datasets and other repos on the huggingface.co hub


In [6]:
info = importlib.metadata.metadata("transformers")

print("Nombre:", info["Name"])
print("Versión:", info["Version"])
print("Email autor:", info["Author-email"])
print("Licencia:", info["License"])
print("Resumen:", info["Summary"])

Nombre: transformers
Versión: 4.33.3
Email autor: transformers@huggingface.co
Licencia: Apache 2.0 License
Resumen: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow


In [7]:
import dspy
from dataclasses import dataclass
from typing import List, Dict
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 1) Para la demo usamos pares (texto, label, breve justificación)
EXAMPLE_BANK = [
    {
        "text": "La delincuencia subió por una sola causa: la migración.",
        "label": "Yes, causal oversimplification",
        "why": "Atribuye fenómeno complejo a una única causa sin considerar otras."
    },
    {
        "text": "El alza de precios se explica por múltiples factores: oferta, demanda, shocks externos y políticas monetarias.",
        "label": "No, there is no causal oversimplification",
        "why": "Reconoce múltiples causas e interacciones."
    },
    {
        "text": "Si la economía va mal, es exclusivamente por el gobierno actual.",
        "label": "Yes, causal oversimplification",
        "why": "Ignora factores globales y estructurales."
    },
    {
        "text": "El desempleo está relacionado con cambios tecnológicos, ciclos, regulación y educación; no hay un único factor.",
        "label": "No, there is no causal oversimplification",
        "why": "Enumera varios factores, evita simplificación."
    },
]

# 2) --------- Indexación semántica con embeddings + FAISS -------------------
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def encode(txts: List[str]) -> np.ndarray:
    embs = embedder.encode(txts, convert_to_numpy=True, normalize_embeddings=True)
    return embs.astype("float32")

bank_texts = [ex["text"] for ex in EXAMPLE_BANK]
bank_embs = encode(bank_texts)

index = faiss.IndexFlatIP(bank_embs.shape[1])   # similitud coseno porque normalizamos
index.add(bank_embs)

def retrieve_similar_examples(query: str, k: int = 2) -> List[Dict]:
    q_emb = encode([query])
    scores, idxs = index.search(q_emb, k)
    return [EXAMPLE_BANK[i] for i in idxs[0]]

# 3) --------- Definimos una Signature DSPy con un campo de “shots” ----------
class OversimplificationJudge(dspy.Signature):
    """
    Decide if the TEXT commits causal oversimplification.
    Return exactly:
    - "Yes, causal oversimplification" OR
    - "No, there is no causal oversimplification"
    Also add a one-sentence justification in Spanish.
    """
    shots = dspy.InputField(desc="Few-shot examples retrieved by RAG")
    text = dspy.InputField()
    answer = dspy.OutputField(desc="Decision + brief justification")

# 4) --------- Módulo DSPy que compone el contexto con los shots -------------
class RAGFewShotCausal(dspy.Module):
    def __init__(self, k=2):
        super().__init__()
        self.k = k
        self.predictor = dspy.Predict(OversimplificationJudge)

    def format_shots(self, examples: List[Dict]) -> str:
        # Formateo claro y compacto para few-shots
        blocks = []
        for ex in examples:
            blocks.append(
                f"""[Ejemplo]
Texto: "{ex['text']}"
Etiqueta: {ex['label']}
Justificación: {ex['why']}"""
            )
        return "\n\n".join(blocks)

    def forward(self, text: str):
        retrieved = retrieve_similar_examples(text, k=self.k)
        shots_str = self.format_shots(retrieved)

        # Llamada al predictor con shots + texto actual
        out = self.predictor(shots=shots_str, text=text)
        return out

# 5) --------- Configuración del LLM para DSPy -------------------------------

lm = dspy.LM('openai/gpt-4o', api_key='coloque su clave aqui')

dspy.configure(lm=lm)

rag = RAGFewShotCausal(k=2)

/home/marcelo/.local/lib/python3.10/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/home/marcelo/.local/lib/python3.10/site-packages/bitsandbytes/cuda_setup/main.py:149: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('/usr/local/cuda-11.8/lib64')}
  warn(msg)



===================================BUG REPORT===================================
Welcome to bitsandbytes. For bug reports, please run

python -m bitsandbytes

 and submit this information together with your error trace to: https://github.com/TimDettmers/bitsandbytes/issues
bin /home/marcelo/.local/lib/python3.10/site-packages/bitsandbytes/libbitsandbytes_cuda120.so
CUDA SETUP: CUDA runtime path found: /usr/local/cuda-12.0/lib64/libcudart.so
CUDA SETUP: Highest compute capability among GPUs detected: 7.5
CUDA SETUP: Detected CUDA version 120
CUDA SETUP: Loading binary /home/marcelo/.local/lib/python3.10/site-packages/bitsandbytes/libbitsandbytes_cuda120.so...


2025-09-15 11:54:31.118517: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/marcelo/.local/lib/python3.10/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [8]:
query = (
    "Un país explica el colapso educativo únicamente por la pandemia, "
    "sin considerar financiamiento, gestión, formación docente u otros factores."
)

pred = rag(text=query)
print("=== PREDICCIÓN ===")
print(pred.answer)


=== PREDICCIÓN ===
Yes, causal oversimplification. La explicación ignora otros factores importantes que contribuyen al colapso educativo.
